In [2]:
import pandas as pd
import numpy as np

# Credit Risk Modeling — Data Understanding

## Objective

The objective of this project is to develop a credit risk model capable of estimating the probability that a borrower experiences serious financial distress within the next two years.

The project will include:

- Data understanding and exploratory analysis
- Data cleaning and preprocessing
- Credit risk feature analysis
- WOE / Information Value
- Logistic Regression
- Model evaluation using ROC-AUC and classification metrics
- Probability of Default estimation
- Expected Loss scenario analysis

## Dataset Overview

The dataset contains historical borrower information including age, income, debt levels, credit utilization, dependents and delinquency behavior.

Target variable:

**SeriousDlqin2yrs**

- 0: No serious financial distress within two years
- 1: Serious financial distress within two years


In [21]:
df = pd.read_csv("../data/raw/cs-training.csv")

df.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [22]:
df.shape

(150000, 12)

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

In [24]:
df.isnull().sum()

Unnamed: 0                                  0
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

In [25]:
df["SeriousDlqin2yrs"].value_counts()


SeriousDlqin2yrs
0    139974
1     10026
Name: count, dtype: int64

In [26]:
df["SeriousDlqin2yrs"].value_counts(normalize=True)

SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64

In [27]:
print("Dimensiones:", df.shape)

print("\nDistribución del target:")
print(df["SeriousDlqin2yrs"].value_counts())

print("\nDistribución porcentual:")
print(df["SeriousDlqin2yrs"].value_counts(normalize=True) * 100)

print("\nValores faltantes:")
print(df.isnull().sum())

print("\nColumnas:")
print(df.columns.tolist())

Dimensiones: (150000, 12)

Distribución del target:
SeriousDlqin2yrs
0    139974
1     10026
Name: count, dtype: int64

Distribución porcentual:
SeriousDlqin2yrs
0    93.316
1     6.684
Name: proportion, dtype: float64

Valores faltantes:
Unnamed: 0                                  0
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

Columnas:
['Unnamed: 0', 'SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans'

In [28]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0]

MonthlyIncome         29731
NumberOfDependents     3924
dtype: int64

In [29]:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage[missing_percentage > 0]

MonthlyIncome         19.820667
NumberOfDependents     2.616000
dtype: float64

In [30]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,150000.0,75000.500000,43301.414527,1.0,37500.750000,75000.500000,112500.250000,150000.0
SeriousDlqin2yrs,150000.0,0.066840,0.249746,0.0,0.000000,0.000000,0.000000,1.0
RevolvingUtilizationOfUnsecuredLines,150000.0,6.048438,249.755371,0.0,0.029867,0.154181,0.559046,50708.0
age,150000.0,52.295207,14.771866,0.0,41.000000,52.000000,63.000000,109.0
NumberOfTime30-59DaysPastDueNotWorse,150000.0,0.421033,4.192781,0.0,0.000000,0.000000,0.000000,98.0
DebtRatio,150000.0,353.005076,2037.818523,0.0,0.175074,0.366508,0.868254,329664.0
MonthlyIncome,120269.0,6670.221237,14384.674215,0.0,3400.000000,5400.000000,8249.000000,3008750.0
NumberOfOpenCreditLinesAndLoans,150000.0,8.452760,5.145951,0.0,5.000000,8.000000,11.000000,58.0
NumberOfTimes90DaysLate,150000.0,0.265973,4.169304,0.0,0.000000,0.000000,0.000000,98.0
NumberRealEstateLoansOrLines,150000.0,1.018240,1.129771,0.0,0.000000,1.000000,2.000000,54.0


In [31]:
df[[
    "MonthlyIncome",
    "NumberOfDependents"
]].describe()

,MonthlyIncome,NumberOfDependents
count,1.202690e+05,146076.000000
mean,6.670221e+03,0.757222
std,1.438467e+04,1.115086
min,0.000000e+00,0.000000
25%,3.400000e+03,0.000000
50%,5.400000e+03,0.000000
75%,8.249000e+03,1.000000
max,3.008750e+06,20.000000


In [32]:
columns_to_check = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents"
]

df[columns_to_check].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
).T

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,99.9%,max
RevolvingUtilizationOfUnsecuredLines,150000.0,6.048438,249.755371,0.0,0.0,0.000000,0.029867,0.154181,0.559046,1.0,1.092956,1571.006,50708.0
age,150000.0,52.295207,14.771866,0.0,24.0,29.000000,41.000000,52.000000,63.000000,78.0,87.000000,94.000,109.0
NumberOfTime30-59DaysPastDueNotWorse,150000.0,0.421033,4.192781,0.0,0.0,0.000000,0.000000,0.000000,0.000000,2.0,4.000000,98.000,98.0
DebtRatio,150000.0,353.005076,2037.818523,0.0,0.0,0.004329,0.175074,0.366508,0.868254,2449.0,4979.040000,10613.074,329664.0
MonthlyIncome,120269.0,6670.221237,14384.674215,0.0,0.0,1300.000000,3400.000000,5400.000000,8249.000000,14587.6,25000.000000,78395.748,3008750.0
NumberOfTimes90DaysLate,150000.0,0.265973,4.169304,0.0,0.0,0.000000,0.000000,0.000000,0.000000,1.0,3.000000,98.000,98.0
NumberOfTime60-89DaysPastDueNotWorse,150000.0,0.240387,4.155179,0.0,0.0,0.000000,0.000000,0.000000,0.000000,1.0,2.000000,98.000,98.0
NumberOfDependents,146076.0,0.757222,1.115086,0.0,0.0,0.000000,0.000000,0.000000,1.000000,3.0,4.000000,6.000,20.0


In [33]:
df["age"].sort_values().head(10)

65695      0
100880    21
118021    21
72428     21
93701     21
70478     21
18278     21
17155     21
99552     21
135342    21
Name: age, dtype: int64

In [34]:
df["RevolvingUtilizationOfUnsecuredLines"].sort_values(ascending=False).head(10)

85489     50708.0
31414     29110.0
16956     22198.0
149160    22000.0
149279    20514.0
117315    18300.0
21978     17441.0
124533    13930.0
72592     13498.0
71705     13400.0
Name: RevolvingUtilizationOfUnsecuredLines, dtype: float64

In [35]:
df["DebtRatio"].sort_values(ascending=False).head(10)

60152     329664.0
36600     326442.0
127047    307001.0
58900     220516.0
4854      168835.0
7513      110952.0
103041    106885.0
69845     101320.0
66785      61907.0
53682      61106.5
Name: DebtRatio, dtype: float64

In [36]:
df["NumberOfTimes90DaysLate"].value_counts().sort_index().tail(15)

NumberOfTimes90DaysLate
4     291
5     131
6      80
7      38
8      21
9      19
10      8
11      5
12      2
13      4
14      2
15      2
17      1
96      5
98    264
Name: count, dtype: int64

In [37]:
df.groupby(
    df["MonthlyIncome"].isnull()
)["SeriousDlqin2yrs"].mean()

MonthlyIncome
False    0.069486
True     0.056137
Name: SeriousDlqin2yrs, dtype: float64

In [38]:
df.groupby(
    df["NumberOfDependents"].isnull()
)["SeriousDlqin2yrs"].mean()

NumberOfDependents
False    0.067410
True     0.045617
Name: SeriousDlqin2yrs, dtype: float64

In [39]:
delinquency_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]

for col in delinquency_cols:
    print(f"\n{col}")
    print(df[col].value_counts().sort_index().tail(10))


NumberOfTime30-59DaysPastDueNotWorse
NumberOfTime30-59DaysPastDueNotWorse
6     140
7      54
8      25
9      12
10      4
11      1
12      2
13      1
96      5
98    264
Name: count, dtype: int64

NumberOfTime60-89DaysPastDueNotWorse
NumberOfTime60-89DaysPastDueNotWorse
3     318
4     105
5      34
6      16
7       9
8       2
9       1
11      1
96      5
98    264
Name: count, dtype: int64

NumberOfTimes90DaysLate
NumberOfTimes90DaysLate
9      19
10      8
11      5
12      2
13      4
14      2
15      2
17      1
96      5
98    264
Name: count, dtype: int64


In [40]:
df[df["age"] == 0]

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
65695,65696,0,1.0,0,1,0.436927,6000.0,6,0,2,0,2.0


In [41]:
df[df["NumberOfTimes90DaysLate"].isin([96, 98])][
    [
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTime60-89DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate",
        "SeriousDlqin2yrs"
    ]
].head(20)

,NumberOfTime30-59DaysPastDueNotWorse,NumberOfTime60-89DaysPastDueNotWorse,NumberOfTimes90DaysLate,SeriousDlqin2yrs
1733,98,98,98,1
2286,98,98,98,0
3884,98,98,98,0
4417,98,98,98,0
4705,98,98,98,0
5073,98,98,98,0
6280,98,98,98,1
7032,98,98,98,1
7117,98,98,98,1
7687,98,98,98,1


In [42]:
for col in delinquency_cols:
    print(
        col,
        df[col].isin([96, 98]).sum(),
        round(df[col].isin([96, 98]).mean() * 100, 4)
    )

NumberOfTime30-59DaysPastDueNotWorse 269 0.1793
NumberOfTime60-89DaysPastDueNotWorse 269 0.1793
NumberOfTimes90DaysLate 269 0.1793


In [43]:
df[df["age"] == 0].T

,65695
Unnamed: 0,65696.000000
SeriousDlqin2yrs,0.000000
RevolvingUtilizationOfUnsecuredLines,1.000000
age,0.000000
NumberOfTime30-59DaysPastDueNotWorse,1.000000
DebtRatio,0.436927
MonthlyIncome,6000.000000
NumberOfOpenCreditLinesAndLoans,6.000000
NumberOfTimes90DaysLate,0.000000
NumberRealEstateLoansOrLines,2.000000


In [44]:
df_clean = df.copy()

In [45]:
df_clean = df_clean.drop(columns=["Unnamed: 0"])

In [46]:
df_clean = df_clean[
    ~df_clean["NumberOfTimes90DaysLate"].isin([96, 98])
]

In [47]:
df_clean = df_clean[df_clean["age"] > 0]

In [48]:
print("Original:", df.shape)
print("Clean:", df_clean.shape)

Original: (150000, 12)
Clean: (149730, 11)


In [49]:
df_clean[
    [
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTime60-89DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate"
    ]
].max()

NumberOfTime30-59DaysPastDueNotWorse    13
NumberOfTime60-89DaysPastDueNotWorse    11
NumberOfTimes90DaysLate                 17
dtype: int64

## Initial Data Quality Findings

Several data quality issues were identified during the initial inspection:

- The dataset contained an index column (`Unnamed: 0`) that does not provide predictive information and was removed.
- One borrower had an age of 0, which is not a valid borrower age. The observation was removed.
- The delinquency variables contained values of 96 and 98 across the same 269 observations (0.1793% of the dataset). These values were treated as anomalous/special-coded observations and removed.
- `MonthlyIncome` contains approximately 19.82% missing values.
- `NumberOfDependents` contains approximately 2.62% missing values.

Missing values will be handled after analyzing their distributions and relationship with the target variable.

In [50]:
print(
    "Utilization > 1:",
    (df_clean["RevolvingUtilizationOfUnsecuredLines"] > 1).sum()
)

print(
    "Utilization > 10:",
    (df_clean["RevolvingUtilizationOfUnsecuredLines"] > 10).sum()
)

print(
    "DebtRatio > 1:",
    (df_clean["DebtRatio"] > 1).sum()
)

print(
    "DebtRatio > 10:",
    (df_clean["DebtRatio"] > 10).sum()
)

print(
    "DebtRatio > 100:",
    (df_clean["DebtRatio"] > 100).sum()
)

Utilization > 1: 3321
Utilization > 10: 241
DebtRatio > 1: 35109
DebtRatio > 10: 28852
DebtRatio > 100: 24377


In [51]:
pd.crosstab(
    df_clean["MonthlyIncome"].isnull(),
    df_clean["DebtRatio"] > 10,
    margins=True
)

DebtRatio,False,True,All
MonthlyIncome,,,
False,118014,2106,120120
True,2864,26746,29610
All,120878,28852,149730


In [52]:
df_clean.groupby(
    df_clean["MonthlyIncome"].isnull()
)["DebtRatio"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
)

,count,mean,std,min,50%,90%,95%,99%,max
MonthlyIncome,,,,,,,,,
False,120120.0,26.631760,424.708591,0.0,0.296344,0.763811,1.129929,663.715,61106.5
True,29610.0,1680.182709,4255.714780,0.0,1170.000000,3791.100000,4906.650000,8086.910,329664.0


In [53]:
df_clean["RevolvingUtilizationOfUnsecuredLines"].describe(
    percentiles=[0.90, 0.95, 0.99, 0.995, 0.999]
)

count    149730.000000
mean          6.057542
std         249.980364
min           0.000000
90%           0.977992
95%           1.000000
99%           1.093343
99.5%         1.368207
99.9%      1572.626000
max       50708.000000
Name: RevolvingUtilizationOfUnsecuredLines, dtype: float64

In [54]:
df_clean.groupby(
    df_clean["RevolvingUtilizationOfUnsecuredLines"] > 1
)["SeriousDlqin2yrs"].agg(["count", "mean"])

,count,mean
RevolvingUtilizationOfUnsecuredLines,,
False,146409,0.059026
True,3321,0.372478


In [55]:
df_clean.groupby(
    df_clean["DebtRatio"] > 10
)["SeriousDlqin2yrs"].agg(["count", "mean"])

,count,mean
DebtRatio,,
False,120878,0.068582
True,28852,0.055074


## Key Data Understanding Findings

- The target variable is highly imbalanced, with approximately 6.68% of borrowers experiencing serious financial distress.
- `MonthlyIncome` has approximately 19.82% missing values.
- `NumberOfDependents` has approximately 2.62% missing values.
- Missing `MonthlyIncome` values are strongly associated with unusually large `DebtRatio` values:
  approximately 90% of borrowers with missing income have `DebtRatio > 10`.
- Borrowers with revolving credit utilization above 100% show a substantially higher default rate than borrowers below that threshold.
- Extreme `DebtRatio` values will not be removed automatically because they appear to be structurally related to missing income.
- Missing-value treatment and outlier handling will be performed during preprocessing after further exploratory analysis.